# Airbnb - Classificacao com Comites de Arvores e Visualizacoes

Este notebook parte da classificacao com arvore visual e adiciona uma comparacao visual entre comites de decisao.

O foco aqui e mostrar, de forma mais clara:
- a performance dos classificadores individuais;
- a melhor performance do comite em paralelo;
- a melhor performance do comite em serie;
- matrizes de confusao;
- graficos comparativos;
- visualizacoes de arvores representativas.

Nao ha etapa de regressao linear neste arquivo. O problema tratado aqui e apenas classificacao das faixas de preco `baixo`, `medio` e `alto`.


In [ ]:
import pandas as pd
import pathlib
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.io as pio
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.model_selection import train_test_split
#pio.renderers.default = 'notebook'
#px.set_mapbox_access_token('YOUR_MAPBOX_ACCESS_TOKEN')


In [ ]:
caminho_bases = pathlib.Path('archive')
base_airbnb = pd.DataFrame()
meses = {'jan' : 1, 'fev' : 2, 'mar' : 3, 'abr' : 4, 'mai' : 5, 'jun' : 6, 'jul' : 7, 'ago' : 8, 'set' : 9, 'out': 10, 'nov' : 11, 'dez' : 12}


for arquivo in caminho_bases.iterdir():
    nome_mes = arquivo.name[:3]
    mes = meses[nome_mes]
    ano = arquivo.name[-8:]
    ano = int(ano.replace('.csv', ''))
    
    df = pd.read_csv(caminho_bases / arquivo.name)
    
    df['ano'] = ano
    df['mes'] = mes

    base_airbnb = base_airbnb._append(df)



- Temos muitas colunas, então vou analisar para usar apenas as colunas necessárias para a previsão dos preços.

- Para isso, vou criar um arquivo em excel com os 1000 primeiros registros e fazer uma analise qualitativa.

- Tipos de informações que eu vou excluir:
-IDS, Links, Informações não relevantes
-Colunas repetidadas ou extremamente parecidas com outras (que dão a mesma informação para o modelo)
-Colunas preenchidas com texto livre (não irá ser rodado nenhuma análise de texto)
-Coluna em que todos ou quase todos os valores são iguais


In [ ]:
colunas = ['host_response_time','host_response_rate','host_is_superhost','host_listings_count','latitude','longitude','property_type','room_type','accommodates','bathrooms','bedrooms','beds','bed_type','amenities','price','security_deposit','cleaning_fee','guests_included','extra_people','minimum_nights','maximum_nights','number_of_reviews','review_scores_rating','review_scores_accuracy','review_scores_cleanliness','review_scores_checkin','review_scores_communication','review_scores_location','review_scores_value','instant_bookable','is_business_travel_ready','cancellation_policy','ano','mes']

base_airbnb = base_airbnb.loc[:, colunas]
display(base_airbnb)

Verificando a quantidade de valores nulos na minha base da dados e fazendo o tratamento 

In [ ]:

for coluna in base_airbnb:
    if base_airbnb[coluna].isnull().sum() > 300000:
        base_airbnb = base_airbnb.drop(coluna, axis=1)
        


In [ ]:
base_airbnb = base_airbnb.dropna()
print(base_airbnb.isnull().sum())

Como preço e extra people estão sendo reconhecidos como objeto, tenho que mudar o tipo de variável da coluna

In [ ]:
print(base_airbnb.dtypes)
print('-'*60)
print(base_airbnb.iloc[0])


In [ ]:
# Certificar-se de que as colunas são strings
base_airbnb['price'] = base_airbnb['price'].astype(str)
base_airbnb['extra_people'] = base_airbnb['extra_people'].astype(str)


# Remover símbolos de dólar e vírgulas, em seguida converter para float
base_airbnb['price'] = base_airbnb['price'].str.replace('$', '').str.replace(',', '').astype(np.float32, copy=False)
base_airbnb['extra_people'] = base_airbnb['extra_people'].str.replace('$', '').str.replace(',', '').astype(np.float32, copy=False)

base_airbnb['price'] = pd.to_numeric(base_airbnb['price'], errors='coerce')
base_airbnb['extra_people'] = pd.to_numeric(base_airbnb['extra_people'], errors='coerce')
print(base_airbnb.dtypes)

#numeric_cols = base_airbnb.select_dtypes(include=[np.number])
#plt.figure(figsize=(15,10))
#sns.heatmap(numeric_cols.corr(), annot=True, cmap='Greens')

Preparando o DataFrame para receber somente valores numéricos na coorelacao 

### Definicao de funcoes para análise de outliers
#### Irei definir funcoes para ajudar na análise de outliers das colunas

In [ ]:
def limites(coluna):
    q1 = coluna.quantile(0.25)
    q3 = coluna.quantile(0.75)
    amplitude = q3 - q1
    return q1 - 1.5 * amplitude, q3 + 1.5 * amplitude
    

def excluirOutliers(df, nome_coluna):
    qtde_linhas = df.shape[0]
    lim_inf, lim_sup = limites(df[nome_coluna])
    df = df.loc[(df[nome_coluna] >= lim_inf) & (df[nome_coluna] <= lim_sup), :]
    linhas_removidas = qtde_linhas - df.shape[0]
    return df, linhas_removidas

## Funcoes de Gráficos

In [ ]:
def diagrama_caixa(coluna):
   fig, (ax1, ax2) =  plt.subplots(1,2)
   fig.set_size_inches(15,5)
   sns.boxplot(x=coluna, ax=ax1)
   ax2.set_xlim(limites(coluna))
   sns.boxplot(x=coluna, ax=ax2)

def histograma(coluna):
   plt.figure(figsize=(15,5))
   sns.distplot(coluna, hist=True)

def grafico_barra(coluna):
   plt.figure(figsize=(15,5))
   ax = sns.barplot(x=coluna.value_counts().index, y=coluna.value_counts())
   ax.set_xlim(limites(coluna))


### Excluindo Outliers

### Price

In [ ]:
base_airbnb, linhas_removidas = excluirOutliers(base_airbnb, 'price')
print(linhas_removidas)
#histograma(base_airbnb['price'])

### Extra people

In [ ]:

base_airbnb, linhas_removidas = excluirOutliers(base_airbnb, 'extra_people')
print(linhas_removidas)
#histograma(base_airbnb['extra_people'])

### Host_listings_count

In [ ]:

base_airbnb, linhas_removidas = excluirOutliers(base_airbnb, 'host_listings_count')
print(linhas_removidas)
#grafico_barra(base_airbnb['host_listings_count'])


### Acommodates

In [ ]:

base_airbnb, linhas_removidas = excluirOutliers(base_airbnb, 'accommodates')
print(linhas_removidas)
#grafico_barra(base_airbnb['accommodates'])

### bathrooms

In [ ]:
base_airbnb, linhas_removidas = excluirOutliers(base_airbnb, 'bathrooms')
print(linhas_removidas)
#grafico_barra(base_airbnb['bathrooms'])

### bedrooms

In [ ]:
base_airbnb, linhas_removidas = excluirOutliers(base_airbnb, 'bedrooms')
print(linhas_removidas)

### Beds

In [ ]:
base_airbnb, linhas_removidas = excluirOutliers(base_airbnb, 'beds')
print(linhas_removidas)
#grafico_barra(base_airbnb['beds'])

### guests_included


Não irei remover os outliers dessa feature, irei remover toda a coluna da análise, pelo que parece, os usuários do airbnb utilizam muito do valor padrão do aplicativo, sendo assim, a coluna "guests_included" irá atrapalhar a análise no geral. Irei desconsiderá-la no modelo.

In [ ]:
#diagrama_caixa(base_airbnb['guests_included'])
#grafico_barra(base_airbnb['guests_included'])
base_airbnb = base_airbnb.drop('guests_included', axis=1)
base_airbnb.shape

### minimum_nights

In [ ]:
#diagrama_caixa(base_airbnb['minimum_nights'])
#grafico_barra(base_airbnb['minimum_nights'])
base_airbnb, linhas_removidas = excluirOutliers(base_airbnb, 'minimum_nights')
print(linhas_removidas)
#grafico_barra(base_airbnb['minimum_nights'])

### maximum_nights

Irei retirar o maximum nights da análise, porque a maioria dos usuários utilizou do valor padrão do aplicativo, e isso influenciaria negativamente na análise que vai ser criada pelo modelo

In [ ]:
base_airbnb = base_airbnb.drop('maximum_nights', axis=1)
base_airbnb.shape

### number_of_reviews

Irei retirar o number_of_reviews, acho que não irá infuenciar na análise

In [ ]:
base_airbnb = base_airbnb.drop('number_of_reviews', axis=1)
base_airbnb.shape

### Tratamento de colunas de texto

### property_type


In [ ]:
#print(base_airbnb['property_type'].value_counts())

plt.figure(figsize=(15,5))
grafico = sns.countplot(x='property_type', data=base_airbnb)
grafico.tick_params(axis= 'x', rotation = 90)

In [ ]:
tabela_tipos_casa = base_airbnb['property_type'].value_counts()
colunas_agrupar = []

for tipo in tabela_tipos_casa.index:
    if tabela_tipos_casa[tipo] < 2000:
        colunas_agrupar.append(tipo)

#print(colunas_agrupar)

for tipo in colunas_agrupar:
    base_airbnb.loc[base_airbnb['property_type']== tipo, 'property_type'] = 'Outros'

#print(base_airbnb['property_type'].value_counts())


plt.figure(figsize=(15,5))
grafico = sns.countplot(x='property_type', data=base_airbnb)
grafico.tick_params(axis= 'x', rotation = 90)


### room_type

In [ ]:
#print(base_airbnb['room_type'].value_counts())

plt.figure(figsize=(15,5))
grafico = sns.countplot(x='room_type', data=base_airbnb)
grafico.tick_params(axis= 'x', rotation = 90)

### bed_type

In [ ]:
#print(base_airbnb['bed_type'].value_counts())

#plt.figure(figsize=(15,5))
#grafico = sns.countplot(x='bed_type', data=base_airbnb)
#grafico.tick_params(axis= 'x', rotation = 90)

# Agrupando categorias de bed_type

tabela_bed_type = base_airbnb['bed_type'].value_counts()
colunas_agrupar = []

for tipo in tabela_bed_type.index:
    if tabela_bed_type[tipo] < 10000:
        colunas_agrupar.append(tipo)

#print(colunas_agrupar)

for tipo in colunas_agrupar:
    base_airbnb.loc[base_airbnb['bed_type']== tipo, 'bed_type'] = 'Outros'

#print(base_airbnb['property_type'].value_counts())


plt.figure(figsize=(15,5))
grafico = sns.countplot(x='bed_type', data=base_airbnb)
grafico.tick_params(axis= 'x', rotation = 90)

#print(base_airbnb['bed_type'].value_counts())

### cancellation_policy

In [ ]:
#print(base_airbnb['cancellation_policy'].value_counts())

#plt.figure(figsize=(15,5))
#grafico = sns.countplot(x='cancellation_policy', data=base_airbnb)
#grafico.tick_params(axis= 'x', rotation = 90)

#agrupando categorias de cancellation_policy 

tabela_cancellation = base_airbnb['cancellation_policy'].value_counts()
colunas_agrupar = []

for tipo in tabela_cancellation.index:
    if tabela_cancellation[tipo] < 10000:
        colunas_agrupar.append(tipo)

#print(colunas_agrupar)

for tipo in colunas_agrupar:
    base_airbnb.loc[base_airbnb['cancellation_policy']== tipo, 'cancellation_policy'] = 'strict'

#print(base_airbnb['property_type'].value_counts())


plt.figure(figsize=(15,5))
grafico = sns.countplot(x='cancellation_policy', data=base_airbnb)
grafico.tick_params(axis= 'x', rotation = 90)

#print(base_airbnb['cancellation_policy'].value_counts())


### amenities

Como temos uma diversidade muito grande, as vezes, as mesmas amenities poderiam ser escritas de forma diferente. Portanto, irei avaliar a quantidade de amenities em cada airbnb para que eu possa usar essa coluna como parametro no modelo.

###

In [ ]:
#print(base_airbnb['amenities'].iloc[0])


base_airbnb['n_amenities'] = base_airbnb['amenities'].str.split(',').apply(len)
base_airbnb = base_airbnb.drop('amenities', axis=1)
base_airbnb.shape

Como tranformei a coluna de amenities, que antes era string em uma coluna com valores numéricos, devo analisar essa nova coluna da mesma forma em que analisei as outras colunas numéricas.

In [ ]:
#diagrama_caixa(base_airbnb['n_amenities'])
#grafico_barra(base_airbnb['n_amenities'])

In [ ]:
base_airbnb, linhas_removidas = excluirOutliers(base_airbnb, 'n_amenities')
print(linhas_removidas)
grafico_barra(base_airbnb['n_amenities'])

## Vizualizacao de Mapa das Propriedades

In [ ]:
amostra = base_airbnb.sample(n=50000)

pio.renderers.default = 'notebook'

centro_mapa = {'lat': amostra.latitude.mean(), 'lon': amostra.longitude.mean()}

mapa = px.density_mapbox(amostra, lat='latitude', lon='longitude', z='price', radius=2.5,
                         center=centro_mapa, zoom=10, 
                         mapbox_style='open-street-map')
mapa.show()



In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

# Configura o renderizador para Jupyter Notebook
pio.renderers.default = 'notebook'

# Defina a chave de API do Mapbox
px.set_mapbox_access_token('YOUR_MAPBOX_ACCESS_TOKEN')

# Exemplo de carregamento de dados (ajuste conforme necessário)
# base_airbnb = pd.read_csv('caminho/para/seus/dados.csv')

# Amostra de 50.000 linhas
amostra = base_airbnb.sample(n=50000)

# Verificando se as colunas necessárias existem
if all(col in amostra.columns for col in ['latitude', 'longitude', 'price']):
    # Calculando o centro do mapa
    centro_mapa = {'lat': amostra.latitude.mean(), 'lon': amostra.longitude.mean()}

    # Criando um mapa de densidade com tiles personalizados do OpenStreetMap
    fig = go.Figure(go.Densitymapbox(
        lat=amostra.latitude, 
        lon=amostra.longitude, 
        z=amostra.price, 
        radius=2.5,
        colorscale='Viridis',
        showscale=True
    ))

    # Adicionando tiles personalizados do OpenStreetMap com relevo
    fig.update_layout(
        mapbox=dict(
            accesstoken='YOUR_MAPBOX_ACCESS_TOKEN',
            style='white-bg',  # Estilo básico para sobrepor com tiles personalizados
            layers=[
                {
                    'sourcetype': 'raster',
                    'source': [
                        "https://tile.opentopomap.org/{z}/{x}/{y}.png"
                    ],
                    'below': 'traces'
                }
            ],
            zoom=10,
            center=go.layout.mapbox.Center(lat=centro_mapa['lat'], lon=centro_mapa['lon'])
        )
    )

    fig.update_layout(margin={"r":0,"t":0,"l":0,"b":0})
    fig.show()

else:
    print("As colunas 'latitude', 'longitude' e 'price' não foram encontradas na amostra.")


## Encoding

Precisamos ajustar as features para facilitar o trabalho do modelo futuro (features de categoria, true e false, etc.)

*Features de valores True ou False, vamos substituir True por 1 e False por 0.


*Features de categoria (features em que os valores da coluna sao textos) vamos utilizar o metodo de encoding de variaveis dummies.

In [ ]:
colunas_tf = ['host_is_superhost', 'instant_bookable', 'is_business_travel_ready']
base_airbnb_cod = base_airbnb.copy()
for coluna in colunas_tf:
    base_airbnb_cod.loc[base_airbnb_cod[coluna] == 't', coluna] = 1
    base_airbnb_cod.loc[base_airbnb_cod[coluna] == 'f', coluna] = 0


In [ ]:
colunas_categorias = ['property_type', 'room_type', 'bed_type', 'cancellation_policy']
base_airbnb_cod = pd.get_dummies(data = base_airbnb_cod, columns = colunas_categorias)
display(base_airbnb_cod.head())

## Modelo de Previsão

In [ ]:
def avaliar_modelo(nome_modelo, y_teste, previsao):
    acuracia = accuracy_score(y_teste, previsao)
    relatorio = classification_report(y_teste, previsao, zero_division=0)
    return (
        f'Modelo {nome_modelo}:\n'
        f'Acurácia: {acuracia:.2%}\n\n'
        f'Relatório de classificação:\n{relatorio}'
    )


### Modelo de classificação

A professora pediu um modelo de árvore simples para classificação. Por isso, este notebook usa apenas `DecisionTreeClassifier`, sem Random Forest, Extra Trees ou outro ensemble.

Como `price` é numérico, ele será convertido em uma categoria de preço:
- `baixo`
- `medio`
- `alto`


In [ ]:
base_modelo = base_airbnb_cod.copy()

base_modelo['categoria_preco'] = pd.qcut(
    base_modelo['price'],
    q=3,
    labels=['baixo', 'medio', 'alto']
)

modelo_arvore = DecisionTreeClassifier(
    max_depth=4,
    min_samples_leaf=30,
    random_state=42
)

y = base_modelo['categoria_preco']
X = base_modelo.drop(['price', 'categoria_preco'], axis=1)

display(base_modelo[['price', 'categoria_preco']].head())
print(y.value_counts())


Separar os dados em treino e teste + treino do modelo de classificação


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

modelo_arvore.fit(X_train, y_train)
previsao = modelo_arvore.predict(X_test)

print(avaliar_modelo('DecisionTreeClassifier', y_test, previsao))


In [ ]:
matriz_confusao = confusion_matrix(y_test, previsao, labels=modelo_arvore.classes_)
disp = ConfusionMatrixDisplay(
    confusion_matrix=matriz_confusao,
    display_labels=modelo_arvore.classes_
)
disp.plot(cmap='Blues')
plt.show()


### Visualização da árvore de decisão

A célula abaixo mostra como a árvore ficou depois do treinamento. Como a árvore foi limitada com `max_depth=4`, ela fica mais simples de interpretar.


In [ ]:
plt.figure(figsize=(28, 14))
plot_tree(
    modelo_arvore,
    feature_names=X_train.columns,
    class_names=modelo_arvore.classes_,
    filled=True,
    rounded=True,
    fontsize=9
)
plt.title('Árvore de decisão para classificação de faixa de preço')
plt.tight_layout()
plt.savefig('arvore_decisao_classificacao.png', dpi=200, bbox_inches='tight')
plt.show()


### Regras da árvore em texto

Esta versão ajuda a explicar a lógica do modelo em formato de regras.


In [ ]:
regras_arvore = export_text(
    modelo_arvore,
    feature_names=list(X_train.columns)
)
print(regras_arvore)


### Modelo escolhido: DecisionTreeClassifier

O modelo escolhido foi uma árvore de decisão simples para classificação. Ele não usa ensemble e foi limitado com `max_depth=4` e `min_samples_leaf=30` para evitar uma árvore muito complexa.

O objetivo agora é classificar o imóvel em uma faixa de preço, não prever o valor exato do preço.


### Importância das variáveis


In [ ]:
importancia_features = pd.DataFrame(
    modelo_arvore.feature_importances_,
    index=X_train.columns,
    columns=['importancia']
)
importancia_features = importancia_features.sort_values(by='importancia', ascending=False)
display(importancia_features)

plt.figure(figsize=(15,5))
ax = sns.barplot(x=importancia_features.index, y=importancia_features['importancia'])
ax.tick_params(axis='x', rotation=90)
plt.show()


### Ajustes finais no modelo

- Remover `is_business_travel_ready`, caso ela não contribua bem para o modelo.


In [ ]:
base_modelo = base_modelo.drop('is_business_travel_ready', axis=1)

y = base_modelo['categoria_preco']
X = base_modelo.drop(['price', 'categoria_preco'], axis=1)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

modelo_arvore.fit(X_train, y_train)
previsao = modelo_arvore.predict(X_test)

print(avaliar_modelo('DecisionTreeClassifier', y_test, previsao))


In [ ]:
base_teste = base_modelo.copy()

for coluna in base_teste:
    if 'bed_type' in coluna:
        base_teste = base_teste.drop(coluna, axis=1)

y = base_teste['categoria_preco']
X = base_teste.drop(['price', 'categoria_preco'], axis=1)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

modelo_arvore.fit(X_train, y_train)
previsao = modelo_arvore.predict(X_test)

print(avaliar_modelo('DecisionTreeClassifier', y_test, previsao))


In [ ]:
dados_modelo = X.copy()
dados_modelo['categoria_preco'] = y
dados_modelo.to_csv('dados_classificacao_arvore_visual.csv', index=False)


## Comites de decisao com visualizacoes

Nesta parte vamos testar comites usando 3 classificadores baseados em arvores:

- `DecisionTreeClassifier`
- `RandomForestClassifier`
- `HistGradientBoostingClassifier`

Depois vamos comparar:

- modelos individuais;
- melhor comite em paralelo;
- melhor comite em serie;
- matriz de confusao de cada comite;
- graficos de metricas;
- arvores representativas para apresentacao.


In [ ]:
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, VotingClassifier, StackingClassifier
from sklearn.metrics import f1_score, precision_score, recall_score
import joblib


### Base final para classificacao

Se voce executar o notebook inteiro, as variaveis `X` e `y` ja estarao prontas. Se quiser rodar apenas esta parte, a celula carrega o CSV salvo na etapa anterior.


In [ ]:
if 'X' not in globals() or 'y' not in globals():
    dados_comite = pd.read_csv('dados_classificacao_arvore_visual.csv')
    y = dados_comite['categoria_preco']
    X = dados_comite.drop('categoria_preco', axis=1)
else:
    dados_comite = X.copy()
    dados_comite['categoria_preco'] = y

print('Formato da base:', X.shape)
display(y.value_counts().rename('quantidade').to_frame())


### Amostra para deixar a aula mais rapida

A base completa e grande. Para testar e apresentar em aula, a amostra de 120 mil linhas costuma ser suficiente. Se quiser usar tudo, troque `usar_amostra_para_comite` para `False`.


In [ ]:
usar_amostra_para_comite = True
qtde_amostra_comite = 120_000

if usar_amostra_para_comite and len(dados_comite) > qtde_amostra_comite:
    dados_treino_comite = dados_comite.sample(
        n=qtde_amostra_comite,
        random_state=42
    )
else:
    dados_treino_comite = dados_comite.copy()

y_comite = dados_treino_comite['categoria_preco']
X_comite = dados_treino_comite.drop('categoria_preco', axis=1)

X_train, X_test, y_train, y_test = train_test_split(
    X_comite,
    y_comite,
    test_size=0.25,
    random_state=42,
    stratify=y_comite
)

print('Treino:', X_train.shape)
print('Teste:', X_test.shape)


### Criacao dos 3 classificadores

Aqui ficam os tres classificadores usados nos comites. Todos sao modelos de classificacao baseados em arvores ou combinacoes de arvores.


In [ ]:
modelo_arvore_comite = DecisionTreeClassifier(
    max_depth=8,
    min_samples_leaf=40,
    random_state=42
)

modelo_random_forest = RandomForestClassifier(
    n_estimators=120,
    max_depth=None,
    min_samples_leaf=20,
    random_state=42,
    n_jobs=-1
)

modelo_gradient_boosting = HistGradientBoostingClassifier(
    max_iter=120,
    learning_rate=0.08,
    random_state=42
)

classificadores_base = [
    ('arvore_decisao', modelo_arvore_comite),
    ('random_forest', modelo_random_forest),
    ('gradient_boosting', modelo_gradient_boosting)
]


### Funcao de avaliacao

A tabela usa quatro metricas. Para escolher o melhor modelo, use principalmente `f1_macro`, porque ela considera as tres classes com o mesmo peso.


In [ ]:
def avaliar_classificador(nome_modelo, tipo_comite, modelo, X_train, X_test, y_train, y_test):
    modelo.fit(X_train, y_train)
    previsao = modelo.predict(X_test)

    metricas = {
        'modelo': nome_modelo,
        'tipo': tipo_comite,
        'acuracia': accuracy_score(y_test, previsao),
        'f1_macro': f1_score(y_test, previsao, average='macro'),
        'precisao_macro': precision_score(y_test, previsao, average='macro', zero_division=0),
        'recall_macro': recall_score(y_test, previsao, average='macro', zero_division=0),
    }

    return metricas, previsao


### Treinamento dos modelos individuais e dos comites


In [ ]:
resultados = []
previsoes = {}
modelos_treinados = {}

modelos_individuais = [
    ('arvore_decisao', 'individual', modelo_arvore_comite),
    ('random_forest', 'individual', modelo_random_forest),
    ('gradient_boosting', 'individual', modelo_gradient_boosting),
]

for nome, tipo, modelo in modelos_individuais:
    metricas, previsao = avaliar_classificador(nome, tipo, modelo, X_train, X_test, y_train, y_test)
    resultados.append(metricas)
    previsoes[nome] = previsao
    modelos_treinados[nome] = modelo

comite_paralelo = VotingClassifier(
    estimators=classificadores_base,
    voting='soft',
    n_jobs=-1
)

metricas, previsao = avaliar_classificador(
    'comite_paralelo_soft_voting',
    'paralelo',
    comite_paralelo,
    X_train,
    X_test,
    y_train,
    y_test
)
resultados.append(metricas)
previsoes['comite_paralelo_soft_voting'] = previsao
modelos_treinados['comite_paralelo_soft_voting'] = comite_paralelo

comite_serie = StackingClassifier(
    estimators=classificadores_base,
    final_estimator=DecisionTreeClassifier(max_depth=4, min_samples_leaf=50, random_state=42),
    stack_method='predict_proba',
    cv=3,
    n_jobs=-1
)

metricas, previsao = avaliar_classificador(
    'comite_serie_stacking',
    'serie',
    comite_serie,
    X_train,
    X_test,
    y_train,
    y_test
)
resultados.append(metricas)
previsoes['comite_serie_stacking'] = previsao
modelos_treinados['comite_serie_stacking'] = comite_serie

comparacao_modelos = pd.DataFrame(resultados).sort_values('f1_macro', ascending=False)
display(comparacao_modelos)


## Visual 1: ranking geral dos modelos

Este grafico mostra quais modelos tiveram melhor resultado. A barra mais importante para decisao e `f1_macro`.


In [ ]:
comparacao_ordenada = comparacao_modelos.sort_values('f1_macro', ascending=True)

plt.figure(figsize=(12, 6))
cores = comparacao_ordenada['tipo'].map({
    'individual': '#8da0cb',
    'paralelo': '#66c2a5',
    'serie': '#fc8d62'
})

plt.barh(comparacao_ordenada['modelo'], comparacao_ordenada['f1_macro'], color=cores)
plt.xlabel('F1 macro')
plt.title('Ranking dos modelos por F1 macro')
plt.xlim(0, 1)
plt.grid(axis='x', alpha=0.25)
plt.show()


## Visual 2: paralelo x serie

Este grafico compara diretamente o melhor comite em paralelo com o melhor comite em serie.


In [ ]:
melhor_paralelo = comparacao_modelos[comparacao_modelos['tipo'] == 'paralelo'].sort_values('f1_macro', ascending=False).iloc[0]
melhor_serie = comparacao_modelos[comparacao_modelos['tipo'] == 'serie'].sort_values('f1_macro', ascending=False).iloc[0]

comparacao_comites = pd.DataFrame([melhor_paralelo, melhor_serie])
display(comparacao_comites)

metricas_plot = ['acuracia', 'f1_macro', 'precisao_macro', 'recall_macro']
comparacao_comites_plot = comparacao_comites.set_index('modelo')[metricas_plot]

ax = comparacao_comites_plot.T.plot(kind='bar', figsize=(12, 6), color=['#66c2a5', '#fc8d62'])
plt.title('Comparacao visual: melhor paralelo x melhor serie')
plt.ylabel('Pontuacao')
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.grid(axis='y', alpha=0.25)
plt.legend(title='Modelo')
plt.show()


## Visual 3: matrizes de confusao lado a lado

Aqui voce consegue mostrar em quais classes cada comite acertou ou errou mais.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, linha, titulo in zip(
    axes,
    [melhor_paralelo, melhor_serie],
    ['Melhor comite em paralelo', 'Melhor comite em serie']
):
    nome_modelo = linha['modelo']
    modelo = modelos_treinados[nome_modelo]
    previsao = previsoes[nome_modelo]
    matriz = confusion_matrix(y_test, previsao, labels=modelo.classes_)

    disp = ConfusionMatrixDisplay(
        confusion_matrix=matriz,
        display_labels=modelo.classes_
    )
    disp.plot(cmap='Blues', ax=ax, colorbar=False)
    ax.set_title(f'{titulo} - {nome_modelo}')

plt.tight_layout()
plt.show()


## Visual 4: desempenho por classe

Este mapa de calor mostra `precision`, `recall` e `f1-score` para cada classe. Ele ajuda a explicar por que a classe `medio` costuma ser a mais dificil.


In [ ]:
def relatorio_em_dataframe(y_real, y_previsto):
    relatorio = classification_report(y_real, y_previsto, output_dict=True, zero_division=0)
    linhas_classes = [classe for classe in ['alto', 'baixo', 'medio'] if classe in relatorio]
    return pd.DataFrame(relatorio).T.loc[linhas_classes, ['precision', 'recall', 'f1-score']]

relatorio_paralelo = relatorio_em_dataframe(y_test, previsoes[melhor_paralelo['modelo']])
relatorio_serie = relatorio_em_dataframe(y_test, previsoes[melhor_serie['modelo']])

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sns.heatmap(relatorio_paralelo, annot=True, fmt='.2f', cmap='YlGnBu', vmin=0, vmax=1, ax=axes[0])
axes[0].set_title('Melhor paralelo')

sns.heatmap(relatorio_serie, annot=True, fmt='.2f', cmap='YlGnBu', vmin=0, vmax=1, ax=axes[1])
axes[1].set_title('Melhor serie')

plt.tight_layout()
plt.show()


## Visual 5: arvore de decisao individual

Esta e a arvore individual usada como um dos classificadores base. Ela foi limitada em profundidade para continuar legivel.


In [ ]:
plt.figure(figsize=(30, 14))
plot_tree(
    modelos_treinados['arvore_decisao'],
    feature_names=X_train.columns,
    class_names=modelos_treinados['arvore_decisao'].classes_,
    filled=True,
    rounded=True,
    fontsize=8,
    max_depth=3
)
plt.title('Arvore de decisao individual - primeiros niveis')
plt.tight_layout()
plt.savefig('visual_arvore_decisao_individual.png', dpi=200, bbox_inches='tight')
plt.show()


## Visual 6: arvore representativa da Random Forest

A Random Forest cria varias arvores. Para apresentacao, podemos visualizar uma delas como exemplo. Ela nao representa a floresta inteira sozinha, mas ajuda a explicar como o modelo toma decisoes.


In [ ]:
arvore_exemplo_rf = modelos_treinados['random_forest'].estimators_[0]

plt.figure(figsize=(30, 14))
plot_tree(
    arvore_exemplo_rf,
    feature_names=X_train.columns,
    class_names=modelos_treinados['random_forest'].classes_,
    filled=True,
    rounded=True,
    fontsize=8,
    max_depth=3
)
plt.title('Exemplo de uma arvore dentro da Random Forest - primeiros niveis')
plt.tight_layout()
plt.savefig('visual_arvore_random_forest_exemplo.png', dpi=200, bbox_inches='tight')
plt.show()


## Visual 7: importancia das variaveis

Este grafico mostra quais variaveis mais pesaram no `RandomForestClassifier`. Ele e util para explicar quais caracteristicas do imovel ajudam mais na classificacao de preco.


In [ ]:
importancias_rf = pd.DataFrame({
    'feature': X_train.columns,
    'importancia': modelos_treinados['random_forest'].feature_importances_
}).sort_values('importancia', ascending=False).head(15)

plt.figure(figsize=(12, 6))
sns.barplot(data=importancias_rf, x='importancia', y='feature', color='#8da0cb')
plt.title('Top 15 variaveis mais importantes - Random Forest')
plt.xlabel('Importancia')
plt.ylabel('Variavel')
plt.grid(axis='x', alpha=0.25)
plt.show()


## Conclusao visual

A escolha final deve ser feita pela maior pontuacao em `f1_macro`. O grafico de ranking mostra o melhor modelo geral, e o grafico paralelo x serie mostra diretamente qual tipo de comite performou melhor.


In [ ]:
melhor_modelo_nome = comparacao_modelos.iloc[0]['modelo']
melhor_modelo = modelos_treinados[melhor_modelo_nome]

print('Melhor modelo geral:', melhor_modelo_nome)
print(comparacao_modelos.iloc[0])

joblib.dump(melhor_modelo, 'modelo_classificacao_comite_visual_arvores.joblib')
comparacao_modelos.to_csv('comparacao_modelos_comite_visual_arvores.csv', index=False)

print('Modelo salvo em: modelo_classificacao_comite_visual_arvores.joblib')
print('Comparacao salva em: comparacao_modelos_comite_visual_arvores.csv')
